# Sesión 1 — Soluciones

> **Instructor:** este notebook contiene las soluciones completas de los ejercicios de sesión 1.  
> Compartir con los estudiantes después de la sesión, no durante.

---

## Setup

In [ ]:
!pip install google-genai numpy Pillow python-dotenv --quiet

from google import genai
from google.genai import types
from pydantic import BaseModel
from typing import Optional, Literal, List
from PIL import Image, ImageDraw
import numpy as np
import json
import io
import os

def load_api_key() -> str:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        key = os.environ.get("GOOGLE_API_KEY")
        if key:
            print("API key cargada desde .env")
            return key
    except ImportError:
        pass
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_API_KEY")
        if key:
            print("API key cargada desde Colab Secrets")
            return key
    except Exception:
        pass
    raise EnvironmentError(
        "No se encontró GOOGLE_API_KEY.\n"
        "Local: archivo .env con GOOGLE_API_KEY=AIza...\n"
        "Colab: panel de Secrets (ícono 🔑) con Notebook access activado."
    )

MODEL = "gemini-3.1-flash-lite-preview"
EMBED_MODEL = "gemini-embedding-2"

GOOGLE_API_KEY = load_api_key()
client = genai.Client(api_key=GOOGLE_API_KEY)

# Schema base del ejercicio 1
class TransactionInfo(BaseModel):
    entity: str
    amount: float
    currency: Literal["PEN", "USD", "EUR", "OTHER"]
    category: str
    bank: Optional[str] = None
    transaction_type: str

print("Setup completo.")

---
## Solución — Ejercicio 1
### Extracción de entidades de transacciones financieras

In [ ]:
transactions = [
    "Transferí S/. 350 a Juan Pérez en BCP para pago de alquiler en Miraflores.",
    "Recibí USD 1,200 desde María García en Bank of America por honorarios de consultoría.",
    "Pagué S/. 89.90 en Saga Falabella San Isidro con tarjeta BBVA, compra de ropa.",
    "Envié S/. 12,500 a Inversiones Andinas SAC RUC 20601234567 por adelanto de contrato TI, desde Interbank.",
    "Tres transferencias internacionales en 48 horas: USD 15,000 a Panamá, USD 12,000 a Islas Caimán, USD 18,000 a Panamá. Total: USD 45,000.",
]

In [ ]:
# === PARTE A — Solución ===

def extract_transaction(text: str, temperature: float = 0.0) -> dict:
    """Extrae datos estructurados de una transacción en texto libre."""
    config = types.GenerateContentConfig(
        temperature=temperature,
        response_mime_type="application/json",
        response_schema=TransactionInfo,
        system_instruction=(
            "Extrae datos de transacciones financieras. "
            "Sé preciso con montos y entidades. "
            "Para transferencias múltiples, usa el monto total."
        )
    )
    response = client.models.generate_content(
        model=MODEL,
        contents=f"Extrae los datos de esta transacción: {text}",
        config=config
    )
    return json.loads(response.text)


# Procesar todas las transacciones con T=0
results = []
for i, txn in enumerate(transactions):
    result = extract_transaction(txn, temperature=0.0)
    results.append(result)
    print(f"TXN-{i+1:03d}: {txn[:60]}...")
    print(f"  → entity={result['entity']}, amount={result['amount']}, currency={result['currency']}")
    print()

print(f"Total procesadas: {len(results)}")

In [ ]:
# === PARTE B — Efecto de temperature ===
# Usamos la TXN-010 (transferencias múltiples sospechosas)

suspicious_txn = transactions[-1]
print(f"Transacción analizada: {suspicious_txn}\n")

print("=== temperature=0.0 (3 ejecuciones) ===")
for run in range(3):
    r = extract_transaction(suspicious_txn, temperature=0.0)
    print(f"  Run {run+1}: amount={r['amount']}, currency={r['currency']}, category={r['category']}")

print()
print("=== temperature=1.0 (3 ejecuciones) ===")
for run in range(3):
    r = extract_transaction(suspicious_txn, temperature=1.0)
    print(f"  Run {run+1}: amount={r['amount']}, currency={r['currency']}, category={r['category']}")

print()
print("Reflexión:")
print("  Con T=0: la respuesta es siempre idéntica. Para extracción de datos en compliance,")
print("  esto es lo que queremos. Un sistema que da respuestas distintas al mismo input")
print("  es imposible de auditar y potencialmente peligroso en producción.")
print("  Con T=1: el campo 'category' puede variar entre ejecuciones aunque el monto sea igual.")
print("  Eso es un riesgo real si esa categoría alimenta un sistema de decisiones.")

---
## Solución — Ejercicio 2
### Voucher → JSON y Circular SBS → artículos con umbrales

In [ ]:
# Schemas del ejercicio 2
class VoucherInternacional(BaseModel):
    amount: float
    currency: str
    beneficiary_bank: str
    beneficiary_name: str
    purpose: str
    status: str

class ThresholdArticle(BaseModel):
    article_number: str
    threshold_value: str
    threshold_unit: str
    description: str

class ArticleList(BaseModel):
    articles: List[ThresholdArticle]

In [ ]:
# === PARTE A — Voucher BBVA → JSON ===

voucher_bbva_text = """
BBVA - SWIFT INTERNACIONAL
Fecha: 28/04/2024 11:05:44
Ref. SWIFT: BSAB20240428PE00192
Tipo: Transferencia internacional
Monto: USD 15,000.00
Equivalente PEN: S/ 56,250.00
T/C aplicado: 3.750
Cuenta origen: 0011-XXXXXXXX-00
Banco beneficiario: Banco Santander Uruguay
SWIFT beneficiario: BSURUS33
Beneficiario: Carlos Mendoza EIRL
Proposito: Pago proveedor materia prima
Estado: EN PROCESO
"""

r_voucher = client.models.generate_content(
    model=MODEL,
    contents=f"Extrae los datos de este voucher bancario: {voucher_bbva_text}",
    config=types.GenerateContentConfig(
        temperature=0.0,
        response_mime_type="application/json",
        response_schema=VoucherInternacional
    )
)

print("=== Parte A — Voucher BBVA ===")
print(json.dumps(json.loads(r_voucher.text), indent=2, ensure_ascii=False))

In [ ]:
# === PARTE B — Circular SBS → artículos con umbrales ===

# Cargar la circular
BASE_DATA = None
for candidate in [
    "data/circulares_sbs/circular_B_2244_2024.md",
    "genai-multimodal-course/data/circulares_sbs/circular_B_2244_2024.md",
]:
    if os.path.exists(candidate):
        BASE_DATA = candidate
        break

if BASE_DATA:
    with open(BASE_DATA, "r") as f:
        circular_text = f.read()
else:
    circular_text = """Artículo 3: umbral USD 10,000 en efectivo, 30 días.\n"""
    print("[Usando texto reducido — sube el archivo para el ejemplo completo]")

r_articles = client.models.generate_content(
    model=MODEL,
    contents=f"""
Del siguiente documento normativo, extrae todos los artículos que mencionan umbrales numéricos.
Un umbral es cualquier monto en USD o PEN, o cualquier plazo en días, horas o meses.

DOCUMENTO:
{circular_text}
""",
    config=types.GenerateContentConfig(
        temperature=0.0,
        response_mime_type="application/json",
        response_schema=ArticleList
    )
)

print("=== Parte B — Artículos con umbrales ===")
parsed = json.loads(r_articles.text)
print(f"Total artículos con umbral: {len(parsed['articles'])}\n")
for art in parsed["articles"]:
    print(f"  Art. {art['article_number']:>3} | {art['threshold_value']:>8} {art['threshold_unit']:<8} | {art['description'][:65]}")

---
## Solución — Caso Práctico
### Compliance sin RAG — similitud coseno manual

In [ ]:
# Función auxiliar
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# Fragmentos de la circular
circular_chunks = [
    """Artículo 3 — Umbral de reporte de operaciones en efectivo.
Las empresas reportarán a la UIF-Perú toda transacción en efectivo que supere USD 10,000 o su equivalente,
en una sola operación o en varias operaciones vinculadas que en conjunto superen dicho monto en 30 días.
Plazo: 15 días hábiles al cierre del mes.""",

    """Artículo 4 — Umbral de reporte de transferencias internacionales.
Reportar toda transferencia internacional que supere USD 10,000 en una sola operación, o cuando la suma
de transferencias del mismo titular al mismo destino supere USD 30,000 en 30 días.
Plazo: 15 días hábiles al cierre del mes.""",

    """Artículo 5 — Monitoreo y alertas automáticas.
Generar alertas cuando un cliente realice más de 5 transferencias internacionales en 48 horas,
independientemente del monto individual. Atender alertas en 3 días hábiles.""",

    """Artículo 6 — Debida diligencia reforzada (DDR).
Se aplica DDR en operaciones con jurisdicciones de alto riesgo GAFI: Panamá e Islas Caimán.
Implica declaración jurada de origen de fondos y actualización del expediente cada 6 meses.""",

    """Artículo 2 — Definición de operación sospechosa.
Operación sospechosa: aquella respecto de la cual no existe justificación razonable o genera indicios
de vinculación con lavado de activos o financiamiento del terrorismo.""",

    """Artículo 7 — Obligaciones del Oficial de Cumplimiento.
Reportar a UIF-Perú operaciones sospechosas dentro de las 24 horas de concluido el análisis,
sin informar al cliente. Mantener registro de alertas por mínimo 10 años.""",

    """Artículo 8 — Sanciones. Multas entre 5 y 200 UIT.
Infracción grave: omisión de reporte sobre el umbral.
Infracción muy grave: encubrimiento de operaciones de LA/FT.""",

    """Artículo 1 — Ámbito de aplicación.
Aplicable a bancos, financieras, cajas municipales, cajas rurales, cooperativas y empresas de servicios de pago.""",

    """Artículo 5b — Detección de fraccionamiento.
Detectar patrones de operaciones fraccionadas para eludir umbrales.
Identificar transacciones que superen el perfil histórico del cliente en más de 3 desviaciones estándar en 90 días.""",

    """Circular B-2251-2024 — Artículo 6 — Dinero electrónico.
Reportar como inusuales: acumulación de más de S/ 1,800 en movimientos diarios en cuentas básicas,
microtransacciones repetitivas (>20 operaciones bajo S/ 10 en 2 horas),
transferencias circulares que regresen al origen en menos de 24 horas."""
]

query = "Un cliente realizó 3 transferencias internacionales en 48 horas por un total de USD 45,000 hacia Panamá e Islas Caimán. ¿Requiere reporte a la UIF?"

# --- Paso 1: Embed chunks y query ---
chunk_result = client.models.embed_content(model=EMBED_MODEL, contents=circular_chunks)
chunk_embeddings = np.array([e.values for e in chunk_result.embeddings])

query_result = client.models.embed_content(model=EMBED_MODEL, contents=query)
query_embedding = np.array(query_result.embeddings[0].values)

# --- Paso 2: Similitud coseno → top-3 ---
similarities = np.array([cosine_similarity(query_embedding, e) for e in chunk_embeddings])
ranked_indices = np.argsort(similarities)[::-1]

print("Rankings:")
for rank, idx in enumerate(ranked_indices[:5]):
    print(f"  {rank+1}. sim={similarities[idx]:.4f} | {circular_chunks[idx][:70].replace(chr(10),' ')}...")

top_3 = [circular_chunks[i] for i in ranked_indices[:3]]

# --- Paso 3: Augmented prompt → Gemini ---
context = "\n\n---\n\n".join(top_3)
augmented_prompt = f"""
Eres un analista de compliance del sistema financiero peruano.
Basándote ÚNICAMENTE en la normativa SBS proporcionada, responde la pregunta. Cita el artículo exacto.
Si la normativa no es suficiente, dilo explícitamente.

=== NORMATIVA SBS ===
{context}

=== PREGUNTA ===
{query}
"""

response = client.models.generate_content(
    model=MODEL,
    contents=augmented_prompt,
    config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=512)
)

print("\n=== RESPUESTA ===")
print(response.text)